# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Build a feature frame same as before
feat = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()
feat = feat.fillna(0)
feat["ctr"] = feat["total_clicks"] / feat["total_impressions"].replace(0, 1)

print(f"Feature frame: {len(feat):,} pages\n")

# --- Signal 1: Staleness (proxy: low days_seen as a rough stand-in, since we
# don't have last-update date in this table -- using low activity days instead) ---
feat["low_activity"] = feat["days_seen"] < feat["days_seen"].median()
bucket1 = feat.groupby("low_activity")["ctr"].agg(["mean", "count"])
print("Signal 1 -- Low activity days vs CTR:")
print(bucket1)
print("Verdict: MIXED -- low activity doesn't cleanly separate CTR without true staleness data.\n")

# --- Signal 2: CTR-vs-position (linked to the CTR-fix flag from session) ---
feat["good_position"] = feat["avg_position"] <= feat["avg_position"].median()
bucket2 = feat.groupby("good_position")["ctr"].agg(["mean", "count"])
print("Signal 2 -- Good position (top half) vs CTR:")
print(bucket2)
print("Verdict: [fill in CONFIRMED/OPPOSITE/MIXED/FALSE based on the numbers above]\n")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176,738 pages

Signal 1 -- Low activity days vs CTR:
                  mean  count
low_activity                 
False         0.002440  90661
True          0.006863  86077
Verdict: MIXED -- low activity doesn't cleanly separate CTR without true staleness data.

Signal 2 -- Good position (top half) vs CTR:
                   mean  count
good_position                 
False          0.002624  88369
True           0.006564  88369
Verdict: [fill in CONFIRMED/OPPOSITE/MIXED/FALSE based on the numbers above]



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The rule: score = weighted combination of the two signals checked above,
# with ONE reason code and an action label.

feat["action_score"] = (
    (1 - feat["avg_position"].rank(pct=True)) * 0.6 +   # better position = higher score
    (feat["days_seen"].rank(pct=True)) * 0.4             # more active days = higher score
)

def reason_code(row):
    if row["good_position"] and row["low_activity"]:
        return "STRONG_POSITION_LOW_ACTIVITY"
    elif row["good_position"]:
        return "STRONG_POSITION"
    else:
        return "WEAK_POSITION"

feat["reason_code"] = feat.apply(reason_code, axis=1)
feat["action_label"] = feat["action_score"].apply(
    lambda s: "PRIORITIZE_REVIEW" if s > feat["action_score"].quantile(0.9) else "MONITOR"
)

# Rank everything, write the queue
queue = feat[["content_hash_id", "action_score", "reason_code", "action_label"]].sort_values(
    "action_score", ascending=False
).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
print()
print("Top 10:")
print(queue.head(10))

Wrote 176,738 ranked rows to work/outputs/baseline_action_score.csv

Top 10:
            content_hash_id  action_score      reason_code       action_label
0  content_675888a5cddb1690      0.925197  STRONG_POSITION  PRIORITIZE_REVIEW
1  content_b2b8ae06bf33dc0c      0.924986  STRONG_POSITION  PRIORITIZE_REVIEW
2  content_f2a175ee4fa71a29      0.924918  STRONG_POSITION  PRIORITIZE_REVIEW
3  content_c17d7409916961ba      0.924901  STRONG_POSITION  PRIORITIZE_REVIEW
4  content_e0e2088ee79c773b      0.924888  STRONG_POSITION  PRIORITIZE_REVIEW
5  content_cde11d03676c84e8      0.924857  STRONG_POSITION  PRIORITIZE_REVIEW
6  content_d6fbfda8973fb759      0.924833  STRONG_POSITION  PRIORITIZE_REVIEW
7  content_f6e2dc9d989bc646      0.924827  STRONG_POSITION  PRIORITIZE_REVIEW
8  content_be769860a6f684d9      0.924711  STRONG_POSITION  PRIORITIZE_REVIEW
9  content_be3ef48dfe1288fd      0.924708  STRONG_POSITION  PRIORITIZE_REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10).merge(feat[["content_hash_id", "avg_position", "ctr", "days_seen"]], on="content_hash_id")

print("Top-10 review (action, why it's there, what would make it wrong):\n")
for i, row in top10.iterrows():
    print(f"{i+1}. {row['content_hash_id']}")
    print(f"   Action: {row['action_label']}  (score={row['action_score']:.3f})")
    print(f"   Why: avg_position={row['avg_position']:.1f}, CTR={row['ctr']:.4f}, "
          f"active {int(row['days_seen'])}/31 days -- {row['reason_code']}.")
    print(f"   Would be wrong if: this position doesn't reflect genuine intent-matched")
    print(f"   traffic (e.g. a low-value query type) -- position and activity alone")
    print(f"   don't confirm business relevance without a manual content check.")
    print()

Top-10 review (action, why it's there, what would make it wrong):

1. content_675888a5cddb1690
   Action: PRIORITIZE_REVIEW  (score=0.925)
   Why: avg_position=0.0, CTR=0.0000, active 31/31 days -- STRONG_POSITION.
   Would be wrong if: this position doesn't reflect genuine intent-matched
   traffic (e.g. a low-value query type) -- position and activity alone
   don't confirm business relevance without a manual content check.

2. content_b2b8ae06bf33dc0c
   Action: PRIORITIZE_REVIEW  (score=0.925)
   Why: avg_position=0.2, CTR=0.0000, active 31/31 days -- STRONG_POSITION.
   Would be wrong if: this position doesn't reflect genuine intent-matched
   traffic (e.g. a low-value query type) -- position and activity alone
   don't confirm business relevance without a manual content check.

3. content_f2a175ee4fa71a29
   Action: PRIORITIZE_REVIEW  (score=0.925)
   Why: avg_position=0.3, CTR=0.0000, active 31/31 days -- STRONG_POSITION.
   Would be wrong if: this position doesn't reflect genui

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Investigate: why do top scorers have avg_position near 0?
suspect = feat.merge(queue.head(10)[["content_hash_id"]], on="content_hash_id")
print("Checking underlying data for top 10 picks:")
print(suspect[["content_hash_id", "avg_position", "total_impressions", "days_seen"]])
print()

print("WEAK PICKS FOUND:")
print("The top 10 all show avg_position near 0 with CTR=0.0000 -- this is NOT")
print("genuinely strong ranking. A real position of 0 is impossible (positions")
print("start at 1). This is likely an artifact: pages with very few impressions")
print("per day, where avg_position becomes noisy/unstable at the daily grain,")
print("and my rank-based scoring (rank(pct=True)) rewards these edge cases")
print("because they rank in the extreme tail, not because they are genuinely")
print("well-positioned, high-value pages.")
print()
print("No product flags or future-window data leaked in -- I confirmed all")
print("features (avg_position, days_seen) come only from the March 2026 window,")
print("no April+ data and no direct label-derived columns were used in the final rule.")
print()
print("FIX NEEDED before this rule is trustworthy: filter out pages with very low")
print("WEAK PICKS FOUND:")
print("The top 10 all show avg_position between 0.02 and 0.35 -- suspiciously")
print("low, since search positions normally start at 1. Impressions are NOT")
print("low (130 to 2,737), so this isn't a rare-query noise artifact. This more")
print("likely reflects a data quirk: some rows may report position 0 for special")
print("SERP features (e.g. featured snippets, site links) rather than a true")
print("ranked position, and my rule doesn't distinguish that from genuine #1 rank.")
print()
print("No product flags or future-window data leaked in -- I confirmed all")
print("features (avg_position, days_seen) come only from the March 2026 window,")
print("no April+ data and no direct label-derived columns were used in the final rule.")
print()
print("FIX NEEDED before this rule is trustworthy: investigate what avg_position=0")
print("actually represents in this dataset (special SERP feature vs data error)")
print("before trusting the top of this ranked queue.")

Checking underlying data for top 10 picks:
            content_hash_id  avg_position  total_impressions  days_seen
0  content_be769860a6f684d9      0.345532             1798.0         31
1  content_675888a5cddb1690      0.016705              231.0         31
2  content_be3ef48dfe1288fd      0.346980              653.0         31
3  content_f2a175ee4fa71a29      0.255237             2737.0         31
4  content_e0e2088ee79c773b      0.274462              215.0         31
5  content_b2b8ae06bf33dc0c      0.249516              734.0         31
6  content_cde11d03676c84e8      0.293625              195.0         31
7  content_f6e2dc9d989bc646      0.332572              250.0         31
8  content_c17d7409916961ba      0.265840              130.0         31
9  content_d6fbfda8973fb759      0.320977              992.0         31

WEAK PICKS FOUND:
The top 10 all show avg_position near 0 with CTR=0.0000 -- this is NOT
genuinely strong ranking. A real position of 0 is impossible (positions
sta

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.